#**WaterDRoP**

This notebook uses the WaterDRoP model to estimate the rate of hydrolysis from a chemical SMILES string. Run it from the WaterDRoP project root directory after installing the required Python dependencies.

Enter a single SMILES string below. Scroll to 'Batch Mode' to enter a file of multiple SMILES strings.

## Enter SMILES string

In [1]:
smiles = 'CNC(=O)ON=CC(C)(C)SC'

### Run the following cells

In [4]:
# Imports
import chemprop
from chemprop import featurizers, data, models
import pandas as pd
import numpy as np
import os
import random
import torch
import lightning
import logging
from lightning import pytorch as pl
from rdkit import Chem
from rdkit.Chem import Draw, DataStructs, Descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG

In [5]:
# Turn off PyTorch messages
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

In [ ]:
# Local project paths
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"

TRAINING_DATA_PATH = DATA_DIR / "training_data.csv"
SAMPLE_INPUT_PATH = DATA_DIR / "sample_input_data.csv"

## Load WaterDRoP

The following cells load the trained WaterDRoP models.

In [ ]:
# Load local WaterDRoP models
filenames_stg1 = [MODEL_DIR / f"wd_stg1_n{i}.pt" for i in range(1, 5)]
filenames_stg2 = [MODEL_DIR / f"wd_stg2_n{i}.pt" for i in range(1, 5)]

required_model_files = filenames_stg1 + filenames_stg2
missing_model_files = [path for path in required_model_files if not path.exists()]
if missing_model_files:
    raise FileNotFoundError(
        "Missing model files:\n" + "\n".join(str(path) for path in missing_model_files)
    )

stg1_models = [models.MPNN.load_from_file(str(path)) for path in filenames_stg1]
stg2_models = [models.MPNN.load_from_file(str(path)) for path in filenames_stg2]

## Generate hydrolysis rate prediction

In [9]:
# Featurize input
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
y = np.random.rand(1)
dset = data.MoleculeDataset([data.MoleculeDatapoint.from_smi(smiles, y)], featurizer)
test_loader = data.build_dataloader(dset, batch_size = 1)

In [10]:
%%capture
# Stage 1 prediction (stable or unstable classification)
predictions = []
for model in stg1_models:
    with torch.inference_mode():
        trainer = pl.Trainer()
        preds = torch.concat(trainer.predict(model, test_loader)).detach().cpu().numpy()
        predictions.append(preds)

predictions = np.array(predictions)
stg1_pred = (predictions.mean(axis=0))[0][0] # average prediction
range_lo_stg1 = (predictions.max(axis=0))[0][0] # lowest model prediction
range_hi_stg1 = (predictions.min(axis=0))[0][0] # highest model prediction

# Stage 2 prediction (half-life of unstable compounds)
predictions = []
for model in stg2_models:
    with torch.inference_mode():
        trainer = pl.Trainer()
        preds = torch.concat(trainer.predict(model, test_loader)).detach().cpu().numpy()
        predictions.append(preds)

predictions = np.array(predictions)
stg2_pred = (predictions.mean(axis=0))[0][0] # average prediction
range_lo_stg2 = (predictions.max(axis=0))[0][0] # lowest model prediction
range_hi_stg2 = (predictions.min(axis=0))[0][0] # highest model prediction

# Final prediction and uncertainty
threshold = np.log(np.log(2)/365)
final_pred = half_life_lo = half_life_hi = unit1 = unit2 = None

if stg1_pred < threshold or stg2_pred < threshold:
    half_life = '> 1 year'
    range_lo = range_lo_stg1
    range_hi = range_hi_stg1
else:
    half_life = f'{float(f"{(np.log(2)/np.exp(stg2_pred)):.2g}"):g}'
    range_lo = range_lo_stg2
    range_hi = range_hi_stg2

if range_lo < threshold:
    half_life_lo = '> 1 year'
else:
    half_life_lo = f'{float(f"{(np.log(2)/np.exp(range_lo)):.2g}"):g}'

if range_hi < threshold:
    half_life_hi = '> 1 year'
else:
    half_life_hi = f'{float(f"{(np.log(2)/np.exp(range_hi)):.2g}"):g}'

unit1 = '' if half_life == '> 1 year' else ('day' if float(half_life) <= 1 else 'days')
unit2 = '' if half_life_lo == '> 1 year' else ('day' if float(half_life_lo) <= 1 else 'days')
unit3 = '' if half_life_hi == '> 1 year' else ('day' if float(half_life_hi) <= 1 else 'days')

## Print estimated hydrolysis half-life

The half-life reflects the average prediction of four trained WaterDRoP models, and the range represents the lowest and highest values of the four predictions.

In [11]:
print('Input SMILES:', smiles)
print('Estimated hydrolysis half-life:', half_life, unit1)
print('Range:', half_life_lo, unit2, 'to', half_life_hi, unit3)

Input SMILES: CNC(=O)ON=CC(C)(C)SC
Estimated hydrolysis half-life: 0.65 day
Range: 0.29 day to 1.6 days


## Similarity search of training data

The following cells display the top matches of a Tanimoto similarity search of the input SMILES string against the data used to train WaterDRoP.

In [ ]:
# Generate fingerprints
training_data = pd.read_csv(TRAINING_DATA_PATH)
smis = training_data['SMILES']
half_lives = training_data['half_life_formatted']

input_mol = Chem.MolFromSmiles(smiles)
training_mols = [Chem.MolFromSmiles(smi) for smi in smis]
fp_generator = GetMorganGenerator(radius=3, fpSize=1024)
input_fp = fp_generator.GetFingerprint(input_mol)
train_fps = [fp_generator.GetFingerprint(mol) for mol in training_mols]

# Tanimoto similarity search
similarities = [(DataStructs.TanimotoSimilarity(train_fp, input_fp), smi, hl) for train_fp, smi, hl in zip(train_fps, smis, half_lives)]

# Top matches in training data
top_matches = sorted(similarities, key=lambda x: x[0], reverse=True)[:2]

mols = [Chem.MolFromSmiles(smiles),Chem.MolFromSmiles(top_matches[0][1]),Chem.MolFromSmiles(top_matches[1][1])]

legends = [f'Input molecule: {Chem.MolToSmiles(input_mol)}\n{half_life} {unit1}',
           f'Match 1: {top_matches[0][1]}\n{top_matches[0][2]}',
           f'Match 2: {top_matches[1][1]}\n{top_matches[1][2]}']

drawer = rdMolDraw2D.MolDraw2DSVG(900, 300, 300, 300)
drawer.DrawMolecules(mols, legends=legends)
drawer.FinishDrawing()

svg = drawer.GetDrawingText()
SVG(svg)

To see the experimental sources of the top matches, search for the SMILES string in the [training dataset](https://github.com/amelielemay/WaterDRoP/blob/main/dataset.xlsx).

# Batch Mode

Set `input_path` below to a local CSV file containing a column titled "SMILES" with one SMILES string per row. By default, the notebook uses `data/sample_input_data.csv`.

In [36]:
input_path = SAMPLE_INPUT_PATH

### Run the following cells

In [37]:
df_input = pd.read_csv(input_path)
input_smis = df_input['SMILES']

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
y = np.random.rand(1)
dset = data.MoleculeDataset([data.MoleculeDatapoint.from_smi(smiles, y)])
test_loader = data.build_dataloader(dset, batch_size = 1)

smiles_column = 'SMILES'
target_columns = ['target']
smis = df_input.loc[:, smiles_column].values
ys = np.random.rand(len(smis))
all_data = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(smis, ys)]
test_dset = data.MoleculeDataset(all_data, featurizer)
test_loader = data.build_dataloader(test_dset, shuffle = False, batch_size = len(smis))

In [38]:
%%capture
# Stage 1 prediction (stable or unstable classification)
predictions = []
for model in stg1_models:
    with torch.inference_mode():
        trainer = pl.Trainer()
        preds = torch.concat(trainer.predict(model, test_loader)).detach().cpu().numpy()
        predictions.append(preds)

predictions = np.array(predictions)
stg1_preds = (predictions.mean(axis=0)) # average predictions
ranges_lo_stg1 = (predictions.max(axis=0)) # lowest model predictions
ranges_hi_stg1 = (predictions.min(axis=0)) # highest model predictions

# Stage 2 prediction (half-life of unstable compounds)
predictions = []
for model in stg2_models:
    with torch.inference_mode():
        trainer = pl.Trainer()
        preds = torch.concat(trainer.predict(model, test_loader)).detach().cpu().numpy()
        predictions.append(preds)

predictions = np.array(predictions)
stg2_preds = (predictions.mean(axis=0)) # average predictions
ranges_lo_stg2 = (predictions.max(axis=0)) # lowest model predictions
ranges_hi_stg2 = (predictions.min(axis=0)) # highest model predictions

# Final prediction and uncertainty
final_preds = []
ranges_lo_str = []
ranges_hi_str = []
threshold = np.log(np.log(2)/365)

for i in range(len(input_smis)):
    if stg1_preds[i] < threshold or stg2_preds[i] < threshold:
        final_preds.append('> 1 year')
        ranges_lo = ranges_lo_stg1[i]
        ranges_hi = ranges_hi_stg1[i]
    else:
        final_preds.append(f'{float(f"{(np.log(2)/np.exp(stg2_preds[i][0])):.2g}"):g}')
        ranges_lo = ranges_lo_stg2[i]
        ranges_hi = ranges_hi_stg2[i]

    if ranges_lo < threshold:
        ranges_lo_str.append('> 1 year')
    else:
        ranges_lo_str.append(f'{float(f"{(np.log(2)/np.exp(ranges_lo[0])):.2g}"):g}')

    if ranges_hi < threshold:
        ranges_hi_str.append('> 1 year')
    else:
        ranges_hi_str.append(f'{float(f"{(np.log(2)/np.exp(ranges_hi[0])):.2g}"):g}')

In [39]:
output = pd.DataFrame({'SMILES': smis, 'Half-life (days)': final_preds, 'Range, low': ranges_lo_str, 'Range, high': ranges_hi_str})

## Half-life predictions

In [40]:
output

,SMILES,Half-life (days),"Range, low","Range, high"
0,CN(C)\C=C\C=C\C=C1/c2ccccc2-c2ccc(cc12)[N+]([O...,11,2.8,25
1,CN1C=C/C(=C/C=C(/C=C/c2cc[n+](C)c3ccccc23)c2cc...,78,45,160
2,CCCN(CCC)c1ccc(C(NC(C)C)=c2ccc(=C(C#N)C#N)cc2)cc1,9.8,5.1,13
3,CC[n+]1c(/C=C/C=C/C=C/N(C)c2ccccc2)sc2c(C)c(C)...,15,10,23
4,CCN1/C(=C/C=C/C=C/C2=C/C(=C/c3sc4ccccc4[n+]3CC...,16,12,28
5,CC[n+]1c(C=CC=C2C=C(C=CN(C)c3ccccc3)CC(C)(C)C2...,9,6.6,11
6,CN(C)c1ccc(/C=C2\CCCc3c2cc(-c2ccccc2)[o+]c3-c2...,55,25,160
7,[B-]1(N2C(=CC(=C2C=C3[N+]1=C(C=C3)CCC(=O)O)C)C...,24,11,57
8,CCN(CC)C1=CC2=C(C=C1)C(=C3C=CC(=[N+](CC)CC)C=C...,51,28,80
